# Digital Crime Investigation
## Notebook 02 — Exploratory Data Analysis (EDA)

**Author:** Baskara Kresna Juniarto  
**Project:** Transaction Fraud & Anomaly Analytics  

---
Phase 2: Understand baseline transaction patterns, user behaviour,
and surface preliminary anomaly signals before formal detection.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:,.0f}'.format)

DATA_DIR = Path('../data')
IMG_DIR  = Path('../images')
IMG_DIR.mkdir(exist_ok=True)

txn      = pd.read_csv(DATA_DIR / 'transactions_clean.csv', parse_dates=['timestamp'])
users    = pd.read_csv(DATA_DIR / 'users.csv')
devices  = pd.read_csv(DATA_DIR / 'devices.csv')
merchants= pd.read_csv(DATA_DIR / 'merchants.csv')

txn_full = txn.merge(users[['user_id','city','account_age_days','risk_score']],
                     on='user_id', how='left', suffixes=('','_home'))
print(f'Dataset loaded: {len(txn_full):,} transactions')

## 1. Monthly Transaction Volume & Value

In [ ]:
monthly = txn.set_index('timestamp').resample('ME').agg(
    txn_count=('transaction_id', 'count'),
    total_amount=('amount', 'sum'),
    unique_users=('user_id', 'nunique'),
    avg_amount=('amount', 'mean'),
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].bar(monthly['timestamp'].dt.strftime('%b %Y'), monthly['txn_count'],
            color='steelblue', edgecolor='white')
axes[0].set_title('Monthly Transaction Count', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Transactions')

axes[1].bar(monthly['timestamp'].dt.strftime('%b %Y'),
            monthly['total_amount'] / 1e9,
            color='darkorange', edgecolor='white')
axes[1].set_title('Monthly Total Value (Rp Billion)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('IDR Billion')

for ax in axes:
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(IMG_DIR / '02_monthly_volume.png', dpi=150, bbox_inches='tight')
plt.show()
monthly

## 2. Transaction Category Breakdown

In [ ]:
cat_stats = txn.groupby('category').agg(
    txn_count=('transaction_id', 'count'),
    total_amount=('amount', 'sum'),
    avg_amount=('amount', 'mean'),
    refund_count=('refund_flag', 'sum'),
).sort_values('txn_count', ascending=False).reset_index()

cat_stats['refund_rate'] = (cat_stats['refund_count'] / cat_stats['txn_count'] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = sns.color_palette('muted', len(cat_stats))
axes[0].barh(cat_stats['category'], cat_stats['txn_count'], color=colors)
axes[0].set_title('Transaction Count by Category', fontweight='bold')
axes[0].set_xlabel('Count')

axes[1].barh(cat_stats['category'], cat_stats['refund_rate'], color=colors)
axes[1].set_title('Refund Rate by Category (%)', fontweight='bold')
axes[1].set_xlabel('Refund Rate %')
for ax in axes:
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(IMG_DIR / '02_category_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
cat_stats

## 3. Hourly Transaction Pattern (Baseline)

In [ ]:
hourly = txn.groupby('txn_hour').agg(
    count=('transaction_id', 'count'),
    refund_count=('refund_flag', 'sum')
).reset_index()
hourly['refund_rate'] = hourly['refund_count'] / hourly['count'] * 100

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.bar(hourly['txn_hour'], hourly['count'], alpha=0.7, color='steelblue', label='Txn Count')
ax2.plot(hourly['txn_hour'], hourly['refund_rate'], 'ro-', label='Refund Rate %', linewidth=2)

ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Transaction Count', color='steelblue')
ax2.set_ylabel('Refund Rate %', color='red')
ax1.set_title('Hourly Transaction Pattern — Count vs Refund Rate', fontweight='bold')
ax1.set_xticks(range(0, 24))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig(IMG_DIR / '02_hourly_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Payment Method Share

In [ ]:
pm_stats = txn.groupby('payment_method').agg(
    count=('transaction_id', 'count'),
    refund_count=('refund_flag', 'sum'),
    avg_amount=('amount', 'mean'),
).reset_index()
pm_stats['refund_rate'] = pm_stats['refund_count'] / pm_stats['count'] * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].pie(pm_stats['count'], labels=pm_stats['payment_method'],
            autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('pastel', len(pm_stats)))
axes[0].set_title('Payment Method Share', fontweight='bold')

axes[1].bar(pm_stats['payment_method'], pm_stats['refund_rate'],
            color=sns.color_palette('muted', len(pm_stats)))
axes[1].set_title('Refund Rate by Payment Method (%)', fontweight='bold')
axes[1].set_ylabel('Refund Rate %')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(IMG_DIR / '02_payment_method.png', dpi=150, bbox_inches='tight')
plt.show()
pm_stats

## 5. City-Level Transaction Heatmap

In [ ]:
city_stats = txn.groupby('location_id').agg(
    txn_count=('transaction_id', 'count'),
    total_amount=('amount', 'sum'),
    unique_users=('user_id', 'nunique'),
    refund_count=('refund_flag', 'sum'),
).reset_index().sort_values('txn_count', ascending=False)
city_stats['refund_rate'] = city_stats['refund_count'] / city_stats['txn_count'] * 100

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(city_stats['location_id'], city_stats['txn_count'],
              color=plt.cm.Blues(city_stats['txn_count'] / city_stats['txn_count'].max()))
ax.set_title('Transaction Count by City', fontsize=14, fontweight='bold')
ax.set_xlabel('City')
ax.set_ylabel('Transaction Count')
ax.tick_params(axis='x', rotation=35)

for bar, rate in zip(bars, city_stats['refund_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{rate:.1f}%', ha='center', va='bottom', fontsize=8, color='darkred')

plt.tight_layout()
plt.savefig(IMG_DIR / '02_city_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
city_stats

## 6. Per-User Behaviour Profile (Top 20 by Spend)

In [ ]:
user_profile = txn_full.groupby('user_id').agg(
    txn_count=('transaction_id', 'count'),
    total_amount=('amount', 'sum'),
    avg_amount=('amount', 'mean'),
    unique_devices=('device_id', 'nunique'),
    unique_cities=('location_id', 'nunique'),
    refund_count=('refund_flag', 'sum'),
    chargeback_count=('chargeback_flag', 'sum'),
    risk_score=('risk_score', 'first'),
).reset_index()
user_profile['refund_rate'] = user_profile['refund_count'] / user_profile['txn_count'] * 100

top20 = user_profile.nlargest(20, 'total_amount')

fig, ax = plt.subplots(figsize=(13, 6))
colors_bar = ['crimson' if r > 10 else 'steelblue' for r in top20['refund_rate']]
ax.bar(top20['user_id'], top20['total_amount'] / 1e6, color=colors_bar)
ax.set_title('Top 20 Users by Total Spend — Red = Refund Rate > 10%',
             fontsize=13, fontweight='bold')
ax.set_xlabel('User ID')
ax.set_ylabel('Total Spend (Rp Million)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(IMG_DIR / '02_top20_users.png', dpi=150, bbox_inches='tight')
plt.show()
top20[['user_id','txn_count','total_amount','refund_rate','unique_devices','unique_cities','risk_score']]

## 7. Correlation: Amount vs Refund / Chargeback

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, flag, label in zip(axes, ['refund_flag','chargeback_flag'],
                            ['Refund', 'Chargeback']):
    data = txn.groupby(flag)['amount'].describe()
    ax.boxplot([txn[txn[flag]==0]['amount'], txn[txn[flag]==1]['amount']],
               labels=[f'No {label}', f'{label}'],
               patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(f'Amount Distribution: {label} vs No {label}', fontweight='bold')
    ax.set_ylabel('Amount (IDR)')
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(
        lambda x, _: f'Rp{x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig(IMG_DIR / '02_amount_vs_flags.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. EDA Summary — Key Findings

In [ ]:
print('=== EDA KEY FINDINGS ===')
print(f"Total transactions          : {len(txn):,}")
print(f"Observation window          : {txn['timestamp'].min().date()} → {txn['timestamp'].max().date()}")
print(f"Unique users                : {txn['user_id'].nunique()}")
print(f"Median transaction amount   : Rp {txn['amount'].median():,.0f}")
print(f"P95 transaction amount      : Rp {txn['amount'].quantile(0.95):,.0f}")
print(f"Overall refund rate         : {txn['refund_flag'].mean()*100:.1f}%")
print(f"Overall chargeback rate     : {txn['chargeback_flag'].mean()*100:.1f}%")
print(f"Most active city            : {txn['location_id'].value_counts().idxmax()}")
print(f"Highest refund rate city    : {city_stats.sort_values('refund_rate', ascending=False).iloc[0]['location_id']}")
print(f"Peak transaction hour       : {hourly.loc[hourly['count'].idxmax(), 'txn_hour']}:00")